In [ ]:
from pathlib import Path
import os

from import_libraries import *  # NOSONAR
from global_variables import *  # NOSONAR
import common_libraries.io_library as io_l

from ml_classification.ml_classification_pipeline import (
    GeospatialClassificationPipeline,
    ClassificationPipelineConfig,
)

In [ ]:
WORK_PATH = Path("C:/work_dir/volvi-2023-2024")

PARCELS_PATH = WORK_PATH / "1_parcel_stats" / "satellite_parcel_time_stats.geoparquet"
OUTPUT_PATH = WORK_PATH / "3_ml_classification"

class_column = "label_gr"
parcel_id_column = "parcel_code"


In [ ]:
gdf_parcels = io_l.read_data(str(PARCELS_PATH))
gdf_parcels.head(3)

In [ ]:
label_counts = gdf_parcels[class_column].value_counts()
rare_labels = label_counts[label_counts < 100].index

gdf_parcels.loc[gdf_parcels[class_column].isin(rare_labels), class_column] = "9999"
print(gdf_parcels[class_column].value_counts())

In [ ]:
gdf_parcels = gdf_parcels.to_crs(2100)
gdf_parcels["x"] = gdf_parcels.centroid.x
gdf_parcels["y"] = gdf_parcels.centroid.y

base_cols = [parcel_id_column, class_column, "geometry"]
extra_features = ["elevation_mean_m", "distance_to_nearest_lake_m", "GAIOIKANOT", "x", "y"]
temporal_features = ("_median", "_sd", "_range",)
original_features = [x for x in gdf_parcels.columns if any(keyword in x for keyword in temporal_features)]

gdf_parcels = gdf_parcels[base_cols + original_features + extra_features].copy()
features = original_features + extra_features

gdf_parcels = gdf_parcels[gdf_parcels[class_column] != 9999].copy()
print(f"Final features for modeling ({len(features)} features): {features}")
display(gdf_parcels.head(3))

In [ ]:
config = ClassificationPipelineConfig(
    project_dir=OUTPUT_PATH,
    target_column=class_column,
    feature_columns=features,
    id_column=parcel_id_column,

    apply_iqr=False,
    spatial_interpolation_method=None,

    cv_folds=5,
    test_size=0.2,
    spatial_split=True,
    spatial_split_grid_size=10,
    spatial_split_method="by_row",

    random_state=SEED_NUMBER,
    # n_jobs=1,

    scoring_primary="f1_macro",
    cv_ranking_method="score_minus_std",
    selection_type="soft_voting",
    top_voting_models=5,
    max_search_candidates=40,
    prediction_confidence_threshold=0.60,

    interpretability_top_models=5,
    interpretability_include_shap=False,

    optimize_class_probabilities=True,
    probability_multiplier_grid=(0.8, 1.0, 1.2, 1.5, 2.0),
    probability_optimization_iterations=2,
    probability_optimization_max_accuracy_drop=0.02,

    rank_confidence_enabled=True,
    force_retrain_models=False,

    label_balancing_method="random_oversample",
    lstm_temporal_statistics=temporal_features,
    selected_models=(
        "bagging",
        "bayesian",
        "decision_tree",
        "extra_trees",
        "random_forest",
        "gradient_boosting",
        "lightgbm",
        "xgboost",
        "knn",
        "neural_network",
        "tensorflow_neural_network",
        "keras_lstm",
   ),
    max_map_geometries=20000,
    open_html_report=True,
)

pipeline = GeospatialClassificationPipeline(config)


In [ ]:
summary_check = pipeline.run_check(df=gdf_parcels)
summary_check

In [ ]:
summary_prepare = pipeline.run_prepare()
summary_prepare

In [ ]:
summary_train = pipeline.run_train()
summary_train

In [ ]:
summary_evaluate = pipeline.run_evaluate()
summary_evaluate

In [ ]:
summary_predict = pipeline.run_predict()
summary_predict

In [ ]:
# Show only rows that were unknown during training
predictions_gdf = pd.read_csv(os.path.join(str(OUTPUT_PATH), "05_predict", "final_predictions_preview.csv"))
predictions_gdf[parcel_id_column] = predictions_gdf[parcel_id_column].astype(str)
print(len(predictions_gdf), "rows in predictions_gdf")
display(predictions_gdf.head())

In [ ]:
# From test results
test_results = pd.read_csv(os.path.join(str(OUTPUT_PATH), "04_evaluate", "confidence", "rank_confidence_test.csv"))
print(len(test_results), "rows in test_results")
test_results["accuracy"] = test_results["true_label"] == test_results["predicted_class"]
accuracy = test_results["accuracy"].mean()
print(f"Accuracy on data: {accuracy:.2%}")

# From final predictions
test_results[parcel_id_column] = test_results[parcel_id_column].astype(str)
test_ids =list(test_results[parcel_id_column].unique())
classified_gdf = predictions_gdf[predictions_gdf[parcel_id_column].isin(test_ids)].copy()
print(len(classified_gdf), "rows in classified_gdf after filtering with test_ids")
classified_gdf["accuracy"] = classified_gdf[class_column] == classified_gdf[f"{class_column}_prediction"]
accuracy = classified_gdf["accuracy"].mean()
print(f"Accuracy on data: {accuracy:.2%}")